# MycoMetrics - Notebook 03: Spoilage & Operations Stats
## Multi-Stage Shrinkage, Shelf-Life Decay, and Environmental Utilities
**Business Focus**: FungiFlow Organics Operational Efficiency & Waste  
**Author**: Data Analytics Team  

### Executive Overview
This notebook breaks down supply chain shrinkage into physical root causes, evaluates inventory lot aging risk in cold storage, and assesses climate electricity cost correlation with production volume.

In [ ]:
import duckdb
import pandas as pd
import scipy.stats as stats

con = duckdb.connect('../data/warehouse.duckdb')
con.execute("SET search_path = 'analytics_analytics'")

waste = con.execute("""
    select 
        loss_stage,
        mushroom_common_name,
        reason_code,
        sum(total_lost_quantity_kg) as total_lost_kg,
        sum(total_estimated_loss_amount) as total_loss_amount
    from mart_inventory_waste
    group by 1, 2, 3
    order by total_lost_kg desc
""").df()

waste.head(10)

### Stage Decomposition of Crop Loss

In [ ]:
stage_summary = waste.groupby('loss_stage')[['total_lost_kg', 'total_loss_amount']].sum().round(2)
stage_summary['pct_of_total_kg_lost'] = (stage_summary['total_lost_kg'] / stage_summary['total_lost_kg'].sum() * 100).round(1)
stage_summary.sort_values(by='total_lost_kg', ascending=False)